# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors Exploration with `mlcroissant`
This notebook demonstrates how to explore and process the FAIR² dataset using the [mlcroissant](https://github.com/mlcommons/croissant) library and the Croissant schema.

### Dataset Source
The dataset metadata is described by a Croissant schema at:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Ensure mlcroissant is installed
!pip install --quiet mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the Croissant dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets and their IDs, along with their fields and columns, referenced by their `@id`.

In [ ]:
# Display the available record sets and their fields by @id
record_sets = list(dataset.record_sets)
print(f"Found {len(record_sets)} record sets:\n")
for rset in record_sets:
    print(f"- Record set @id: {rset['@id']}")
    name = rset.get('name', '(no name)')
    print(f"  Name: {name}")
    fields = rset.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    elif isinstance(fields, str):
        fields = [fields]
    if fields:
        print("  Fields:")
        for f in fields:
            if isinstance(f, dict):
                print(f"    - {f.get('@id', f)}")
            else:
                print(f"    - {f}")
    else:
        print("  (No fields found)")
    print("")

## 3. Data Extraction
Load data from each record set available into a DataFrame for analysis.

Below, we use the record set `@id` and field `@id` values from the overview above.

In [ ]:
# Collect all record set @ids
record_set_ids = [rset['@id'] for rset in dataset.record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    # Load all records from this record set
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded DataFrame for record set: {record_set_id}, shape: {df.shape}")
    else:
        print(f"No records found for record set: {record_set_id}")

# For demonstration, choose the first loaded DataFrame
if dataframes:
    chosen_record_set_id = next(iter(dataframes.keys()))
    print(f"\nColumns in DataFrame for record set {chosen_record_set_id}:")
    print(dataframes[chosen_record_set_id].columns.tolist())
    display(dataframes[chosen_record_set_id].head())
else:
    print("No tabular dataframes were loaded.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps – filtering, normalizing, grouping – with all references by `@id`.

Let us choose a numeric field (with its `@id`) and demonstrate filtering, normalization, and grouping.

In [ ]:
# ---- Example configuration ----
# If the dataset includes a numeric field like 'Age' referenced by its @id, e.g. 'https://api.app.sen.science/frontiers/7862866/field/age', set below.
# Please replace these @ids as needed based on the output in the previous step.

import numpy as np

if dataframes:
    # Pick record set and fields by @id (example @id chosen below, replace with yours if needed)
    record_set_id = chosen_record_set_id
    df = dataframes[record_set_id]
    # Try to automatically choose a numeric field, e.g. Age or Interval fields
    numeric_candidates = [col for col in df.columns if df[col].dtype in [float, int, np.float64, np.int64]]
    if not numeric_candidates:
        # Or try simple string matching for "age" or "interval" fields
        numeric_candidates = [col for col in df.columns if 'age' in col.lower() or 'interval' in col.lower()]
    
    if numeric_candidates:
        numeric_field_id = numeric_candidates[0]
        print(f"Using numeric field: {numeric_field_id}")

        # Remove missing values
        filtered_df = df[pd.to_numeric(df[numeric_field_id], errors='coerce').notnull()].copy()
        filtered_df[numeric_field_id] = filtered_df[numeric_field_id].astype(float)
        threshold = filtered_df[numeric_field_id].mean()
        # Filter for values greater than the mean
        filtered_above_mean = filtered_df[filtered_df[numeric_field_id] > threshold]
        print(f"Filtered rows where {numeric_field_id} > {threshold:.2f}:")
        display(filtered_above_mean[[numeric_field_id]].head())

        # Normalize
        col_norm = f"{numeric_field_id}_normalized"
        filtered_above_mean[col_norm] = (filtered_above_mean[numeric_field_id] - filtered_above_mean[numeric_field_id].mean()) / filtered_above_mean[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_above_mean[[numeric_field_id, col_norm]].head())

        # Try grouping by a categorical field, e.g. "Sex" or "AnatomicalLocation" (by @id)
        group_field_candidates = [col for col in df.columns if any(x in col.lower() for x in ['sex', 'site', 'location', 'group'])]
        if group_field_candidates:
            group_field_id = group_field_candidates[0]
            print(f"Grouping by field: {group_field_id}")
            grouped = filtered_above_mean.groupby(group_field_id)[numeric_field_id].mean()
            print(f"Group mean of {numeric_field_id} by {group_field_id} (first few):")
            display(grouped.head())
        else:
            print("No suitable group field found to perform grouping.")
    else:
        print("No numeric field found for EDA.")
else:
    print("No DataFrame available for EDA.")

## 5. Visualization
Visualize the distribution of a numeric variable (by `@id`) and relationships if grouping is possible.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and 'numeric_field_id' in locals():
    # Histogram of numeric field
    plt.figure(figsize=(8,5))
    sns.histplot(df[numeric_field_id].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    # If grouping field available
    if 'group_field_id' in locals():
        plt.figure(figsize=(8,5))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=30)
        plt.show()
else:
    print("Not enough information for visualization.")

## 6. Conclusion
In this notebook, we loaded the FAIR² clinical dataset defined by a Croissant schema using the `mlcroissant` library, explored its structure by referencing all entities with their `@id`, and performed basic data filtering, normalization, grouping, and visualization. This approach provides a reproducible template for processing FAIR datasets in biomedical research following best practices for referencing data elements by their unique identifiers.